# Reproduce the study (Google Colab GPU or a local machine)

Runs the same pipeline as `python scripts/run_all.py`. All logic lives in `src/kidney_tl/`, so results match a command-line run.

* **Colab:** upload this repository folder to Google Drive (or clone it from GitHub), choose *Runtime → Change runtime type → T4 GPU*, then *Run all*.
* **Local:** open the notebook from the `notebooks/` folder; it finds the repository automatically.

Set `SMOKE = True` for a few-minute check of every stage on synthetic data.

In [ ]:
SMOKE = False      # True: run every stage on tiny synthetic data (minutes), no dataset download
DRIVE_REPO = "/content/gdrive/MyDrive/kidney-ct-transfer-learning"   # Colab only: where the repository lives

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    from google.colab import drive  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    drive.mount("/content/gdrive")
    REPO = Path(DRIVE_REPO)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub", "huggingface_hub", "pyarrow"], check=True)
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)
THREADS, NICE = ("0", "0") if ON_COLAB else ("6", "10")   # no limits on a dedicated GPU runtime
print("Repository:", REPO, "| Colab:", ON_COLAB)

In [ ]:
def run(*args):
    result = subprocess.run([sys.executable, *args], capture_output=True, text=True)
    print(result.stdout[-3000:])
    if result.returncode != 0:
        print(result.stderr[-3000:])
        raise RuntimeError(f"{args[0]} failed")

if SMOKE:
    run("scripts/run_all.py", "--smoke", "--threads", "2", "--nice", NICE)
else:
    if not (REPO / "data" / "ct-kidney").exists():
        run("scripts/download_data.py", "--source", "kaggle")
    run("scripts/run_all.py", "--data-dir", "data/ct-kidney", "--threads", THREADS, "--nice", NICE)

In [ ]:
from IPython.display import Markdown, display

if not SMOKE:
    display(Markdown((REPO / "results" / "REPORT.md").read_text()))
    display(Markdown((REPO / "results" / "IMPROVEMENTS.md").read_text()))
else:
    print("Smoke run finished; reports were written to a temporary folder (see the output above).")